# 05 · Classification supervisée des images

**Ce que fait ce notebook.** Puisque l'information est présente dans les photographies, que devient la
performance lorsqu'on entraîne réellement un modèle à prédire la catégorie ? Le notebook compare
plusieurs stratégies de data augmentation sur le jeu de validation, puis n'ouvre le jeu réservé
qu'une fois, avec la seule stratégie retenue.

**Ce qu'il établit.** 137 produits sur 158 correctement classés à partir des seules images. La data
augmentation n'apporte pas d'amélioration nette, mais elle déplace les erreurs d'une catégorie à
l'autre.

In [1]:
import sys

sys.path.insert(0, "..")
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

## Protocole

Tant qu'on cherchait à savoir si des groupes existaient, tout le corpus pouvait servir. Il faut à
présent réserver des produits que le modèle ne verra pas pendant son apprentissage.

La découpe se fait en trois parts et non en deux. Comparer plusieurs stratégies directement sur le
jeu de test, puis retenir la meilleure, ferait de son score une mesure de la qualité de notre
sélection autant que de celle du modèle. Le jeu de validation existe pour absorber ces comparaisons.

In [2]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

from src.pipeline import LABEL_COL, load, split
from src.supervise_image import extraire, tete

df = load()
train, val, test = split(df)
enc = LabelEncoder().fit(df[LABEL_COL])
etiquettes = list(enc.classes_)
y_tr, y_va, y_te = (enc.transform(d[LABEL_COL]) for d in (train, val, test))
ids_tr, ids_va, ids_te = (d["uniq_id"].tolist() for d in (train, val, test))

print(f"entraînement {len(ids_tr)} · validation {len(ids_va)} · réservé {len(ids_te)}")

entraînement 735 · validation 157 · réservé 158


## Modèle

VGG16 conserve ses poids d'origine et sert d'extracteur ; seule une petite tête de classification
est apprise par-dessus. Réajuster les 138 millions de paramètres du réseau sur 735 images le
conduirait à apprendre ces images plutôt que la tâche.

Le fonctionnement tient en une phrase : la photographie entre dans le réseau figé, qui en produit
512 nombres ; la tête reçoit ces 512 nombres et rend 7 probabilités.

In [3]:
X_tr, _ = extraire(ids_tr)
X_va, _ = extraire(ids_va)
X_te, _ = extraire(ids_te)
print(f"chaque image devient {X_tr.shape[1]} nombres")

chaque image devient 512 nombres


## Article de référence

Elle appartient au jeu réservé : le modèle ne l'a jamais vue.

In [4]:
montre = test[test["product_name"].str.contains("V9 METAL STRAP", na=False)]
X_montre, _ = extraire(montre["uniq_id"].tolist())

modele_simple = tete().fit(X_tr, y_tr)
probabilites = modele_simple.predict_proba(X_montre)[0]

for categorie, p in sorted(zip(etiquettes, probabilites), key=lambda t: -t[1]):
    print(f"  {categorie:28s} {p:.3f}")

  Watches                      0.977
  Home Decor & Festive Needs   0.011
  Kitchen & Dining             0.007
  Baby Care                    0.003
  Beauty and Personal Care     0.001
  Computers                    0.001
  Home Furnishing              0.000


## Data augmentation

On fabrique de nouvelles images d'entraînement en transformant celles dont on dispose. Le choix des
transformations est contraint par la nature des photographies : des produits de catalogue, centrés
et cadrés de la même manière. Un retournement horizontal ou une légère rotation restent plausibles ;
un retournement vertical produirait des images qu'on ne rencontrera jamais.

Quatre stratégies sont comparées **sur la validation**, pas sur le jeu réservé.

In [5]:
strategies = [
    ("Sans augmentation", "aucune", 0),
    ("Augmentation douce ×4", "douce", 4),
    ("Augmentation forte ×4", "forte", 4),
    ("Augmentation forte ×8", "forte", 8),
]

selection, par_classe, modeles = [], {}, {}
for nom, intensite, n in strategies:
    if n == 0:
        X, y = X_tr, y_tr
    else:
        X_aug, index = extraire(ids_tr, intensite=intensite, copies=n)
        X, y = np.vstack([X_tr, X_aug]), np.concatenate([y_tr, y_tr[index]])

    clf = tete().fit(X, y)
    pred = clf.predict(X_va)
    modeles[nom] = clf
    scores = f1_score(y_va, pred, average=None, labels=range(len(etiquettes)))
    par_classe[nom] = dict(zip(etiquettes, scores.round(3)))
    selection.append(
        {
            "Stratégie": nom,
            "Images d'entraînement": int(X.shape[0]),
            "F1 macro (validation)": round(float(f1_score(y_va, pred, average="macro")), 4),
        }
    )

pd.DataFrame(selection).sort_values("F1 macro (validation)", ascending=False)

,Stratégie,Images d'entraînement,F1 macro (validation)
1,Augmentation douce ×4,3675,0.8277
2,Augmentation forte ×4,3675,0.8265
0,Sans augmentation,735,0.8216
3,Augmentation forte ×8,6615,0.8149


Sur la validation, l'augmentation douce obtient le meilleur score. Le gain, six millièmes de point
de F1 macro, soit **moins d'un produit sur 157**, est trop faible pour conclure à une amélioration
nette. Une augmentation plus forte et répétée dégrade en revanche nettement la performance.

Une explication possible tient à la nature très standardisée des photographies : multiplier les
transformations artificielles peut éloigner les images d'entraînement de la distribution réellement
observée. C'est une hypothèse plausible, que ces quatre essais ne démontrent pas.

Le résultat est plus intéressant catégorie par catégorie.

In [6]:
pd.DataFrame(par_classe)

,Sans augmentation,Augmentation douce ×4,Augmentation forte ×4,Augmentation forte ×8
Baby Care,0.750,0.762,0.762,0.818
Beauty and Personal Care,0.810,0.800,0.810,0.829
Computers,0.810,0.809,0.783,0.750
Home Decor & Festive Needs,0.739,0.776,0.792,0.773
Home Furnishing,0.905,0.884,0.837,0.818
Kitchen & Dining,0.920,0.913,0.939,0.898
Watches,0.818,0.851,0.864,0.818


*Baby Care*, la catégorie la plus fragile, gagne près de sept points à mesure que l'augmentation
s'intensifie. *Computers* et *Home Furnishing* suivent le chemin inverse. La moyenne ne bouge pas
parce que les gains et les pertes se compensent : **l'augmentation déplace les erreurs plutôt
qu'elle ne les supprime**.

Cela suggère qu'une augmentation générique, appliquée uniformément, n'est probablement pas la bonne
stratégie.

## Jeu réservé · une seule ouverture

In [7]:
retenue = max(selection, key=lambda r: r["F1 macro (validation)"])["Stratégie"]
pred_te = modeles[retenue].predict(X_te)

print(f"stratégie retenue : {retenue}")
print(f"F1 macro          : {f1_score(y_te, pred_te, average='macro'):.4f}")
print(f"exactitude        : {accuracy_score(y_te, pred_te):.4f}")
print(
    f"                    {int(accuracy_score(y_te, pred_te) * len(y_te))}/{len(y_te)} bien classés"
)

stratégie retenue : Augmentation douce ×4
F1 macro          : 0.8671
exactitude        : 0.8671
                    137/158 bien classés


In [8]:
confusion = pd.crosstab(
    pd.Series([etiquettes[i] for i in y_te], name="réelle"),
    pd.Series([etiquettes[i] for i in pred_te], name="prédite"),
)
confusion

prédite,Baby Care,Beauty and Personal Care,Computers,Home Decor & Festive Needs,Home Furnishing,Kitchen & Dining,Watches
réelle,,,,,,,
Baby Care,19,0,0,1,2,0,0
Beauty and Personal Care,1,18,0,0,1,1,1
Computers,0,0,17,3,0,3,0
Home Decor & Festive Needs,1,0,0,21,0,1,0
Home Furnishing,2,0,1,1,19,0,0
Kitchen & Dining,0,0,0,1,0,21,0
Watches,0,0,1,0,0,0,22


Deux *Baby Care* sont prédits *Home Furnishing*, et deux *Home Furnishing* sont prédits *Baby Care*.
C'est la confusion que le regroupement sans étiquettes avait déjà fait apparaître au notebook 04.

Sa réapparition dans deux approches très différentes suggère qu'elle ne dépend pas uniquement du
modèle employé, et qu'il existe une ambiguïté réelle entre certaines images de ces deux catégories.
La supervision la réduit sans la faire disparaître.

**Ce que ce notebook établit.** L'image seule porte une part importante de l'information : 137 produits
sur 158, avec un réseau dont aucun poids n'a été réentraîné et sans utiliser une ligne de
description.